# 00. Prepare independent test set

**Purpose**: Create a held-out independent parent set for final benchmark results.

This addresses Reviewer 2's request to "rerun the analysis with an independent
set of parent molecules."

**Design rules**:
- Same drug-like molecule selection criteria as the development set (01_prepare_parents)
- ChEMBL canonical SMILES overlap with the existing 1000-parent development set is excluded
- Parent selection rules and random seed are fixed before benchmark execution
- All 4 methods use the same independent test set
- Run this notebook **once** to fix the test set; do not regenerate during analysis

**Output**: `data/chembl_independent_test.csv`


In [ ]:
import sys
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import (
    N_PARENTS, RANDOM_SEED,
    MW_MIN, MW_MAX, HBD_MAX, HBA_MAX, ROTB_MAX, ALOGP_MAX,
    N_PARENT_CANDIDATES_MULTIPLIER, SAVE_EVERY,
    PARENT_DATASET, INDEPENDENT_TEST_DATASET,
)
from utils.descriptors import calc_descriptors
from utils.scaffold import get_murcko_scaffold_smiles

# Use a different seed for independent test set collection to get a different
# slice of ChEMBL, but same selection criteria.
TEST_RANDOM_SEED = RANDOM_SEED + 1000   # 1042
N_TEST_PARENTS   = N_PARENTS             # same size as development set (1000)

print(f"Development set : {PARENT_DATASET}")
print(f"Independent test: {INDEPENDENT_TEST_DATASET}")
print(f"Test random seed: {TEST_RANDOM_SEED}")
print(f"Target size     : {N_TEST_PARENTS}")


## Step 1: Load development set (to exclude overlapping SMILES)

In [ ]:
dev_df = pd.read_csv(PARENT_DATASET)
# Support both old ('smiles') and new ('canonical_smiles') column names
smiles_col = "canonical_smiles" if "canonical_smiles" in dev_df.columns else "smiles"
dev_smiles = set(dev_df[smiles_col].dropna().unique())
print(f"Development set: {len(dev_df)} rows, {len(dev_smiles)} unique SMILES to exclude")


## Step 2: Collect from ChEMBL

Same physicochemical filters as the development set.
Canonical SMILES already in the development set are excluded.


In [ ]:
from chembl_webresource_client.settings import Settings
Settings.Instance().TIMEOUT = 60
Settings.Instance().TOTAL_RETRIES = 10
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
results = molecule.filter(
    molecule_properties__mw_freebase__gte=MW_MIN,
    molecule_properties__mw_freebase__lte=MW_MAX,
    molecule_properties__hbd__lte=HBD_MAX,
    molecule_properties__hba__lte=HBA_MAX,
    molecule_properties__rtb__lte=ROTB_MAX,
    molecule_properties__alogp__lte=ALOGP_MAX,
    molecule_type="Small molecule",
).only(["molecule_chembl_id", "molecule_structures", "molecule_properties"])


In [ ]:
import os
from tqdm import tqdm

INTERMEDIATE_FILE = "../data/chembl_test_candidates.csv"

if os.path.exists(INTERMEDIATE_FILE):
    df_saved = pd.read_csv(INTERMEDIATE_FILE)
    records = df_saved.to_dict("records")
    seen_smiles = set(df_saved["smiles"].dropna()) | dev_smiles
    print(f"Resuming from {len(records)} saved candidates.")
else:
    records = []
    seen_smiles = set(dev_smiles)   # pre-exclude development set
    print("Starting fresh.")

target = N_TEST_PARENTS * N_PARENT_CANDIDATES_MULTIPLIER

try:
    for mol in tqdm(results, desc="Collecting"):
        if len(records) >= target:
            break
        try:
            smi = mol.get("molecule_structures", {}).get("canonical_smiles")
            cid = mol.get("molecule_chembl_id")
            if not smi or not cid:
                continue
            rdmol = Chem.MolFromSmiles(smi)
            if rdmol is None:
                continue
            canon = Chem.MolToSmiles(rdmol, canonical=True, isomericSmiles=True)
            if canon in seen_smiles:
                continue
            seen_smiles.add(canon)
            desc = calc_descriptors(rdmol)
            records.append({
                "chembl_id": cid,
                "smiles":    canon,
                "mw":        Descriptors.MolWt(rdmol),
                "logp":      Descriptors.MolLogP(rdmol),
                **desc,
            })
            if len(records) % SAVE_EVERY == 0:
                pd.DataFrame(records).to_csv(INTERMEDIATE_FILE, index=False)
                print(f"Saved: {len(records)}")
        except Exception:
            continue
except Exception as e:
    print(f"Interrupted: {e}")
finally:
    pd.DataFrame(records).to_csv(INTERMEDIATE_FILE, index=False)
    print(f"Total collected: {len(records)}")


## Step 3: Filter for valid Murcko scaffold and sample N_TEST_PARENTS

In [ ]:
df_all = pd.read_csv(INTERMEDIATE_FILE)
df_all = df_all.drop_duplicates(subset="smiles").reset_index(drop=True)
print(f"Before scaffold filter: {len(df_all)}")

def _scaffold_smi(smi):
    mol = Chem.MolFromSmiles(smi)
    return get_murcko_scaffold_smiles(mol) if mol else None

df_all["murcko_scaffold_smiles"] = df_all["smiles"].apply(_scaffold_smi)
df_valid = df_all.dropna(subset=["murcko_scaffold_smiles"]).reset_index(drop=True)
print(f"After scaffold filter:  {len(df_valid)}")

if len(df_valid) < N_TEST_PARENTS:
    raise RuntimeError(
        f"Only {len(df_valid)} candidates with valid scaffold. "
        f"Need {N_TEST_PARENTS}. Increase multiplier or relax filters."
    )

df = df_valid.sample(n=N_TEST_PARENTS, random_state=TEST_RANDOM_SEED).reset_index(drop=True)
print(f"Sampled: {len(df)}")


## Step 4: Build final CSV

In [ ]:
df = df.rename(columns={"chembl_id": "parent_id", "smiles": "canonical_smiles"})

# Recalculate descriptors for safety
def _desc(smi):
    mol = Chem.MolFromSmiles(smi)
    return calc_descriptors(mol) if mol else {"hbd":None,"hba":None,"ar":None,"rotb":None}

descs = df["canonical_smiles"].apply(_desc).apply(pd.Series)
for col in descs.columns:
    df[col] = descs[col]

cols_front = ["parent_id","canonical_smiles","murcko_scaffold_smiles",
              "hbd","hba","ar","rotb","mw","logp"]
df = df[[c for c in cols_front if c in df.columns] +
        [c for c in df.columns if c not in cols_front]]

df.to_csv(INDEPENDENT_TEST_DATASET, index=False)
print(f"Saved {len(df)} independent test parents → {INDEPENDENT_TEST_DATASET}")


## Step 5: Verify no overlap with development set

In [ ]:
dev_df  = pd.read_csv(PARENT_DATASET)
test_df = pd.read_csv(INDEPENDENT_TEST_DATASET)

dev_smiles_col  = "canonical_smiles" if "canonical_smiles" in dev_df.columns else "smiles"
test_smiles_col = "canonical_smiles"

dev_set  = set(dev_df[dev_smiles_col].dropna())
test_set = set(test_df[test_smiles_col].dropna())
overlap  = dev_set & test_set

print(f"Development set : {len(dev_set)} unique SMILES")
print(f"Independent test: {len(test_set)} unique SMILES")
print(f"Overlap         : {len(overlap)}")
assert len(overlap) == 0, f"Overlap detected: {len(overlap)} shared SMILES!"
print("✓ No overlap — test set is independent.")
print()
print("Null scaffold count:", test_df["murcko_scaffold_smiles"].isna().sum())
assert test_df["murcko_scaffold_smiles"].isna().sum() == 0
assert len(test_df) == N_TEST_PARENTS
print(f"✓ {len(test_df)} parents, all with valid Murcko scaffold.")
test_df[["parent_id","canonical_smiles","murcko_scaffold_smiles","hbd","hba","ar","rotb"]].head()
